In [ ]:
import sys, os, random
sys.path.append(os.path.join(os.getcwd(), 'CONFOLD')) #add CONFOLD to path

import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import pandas as pd

from foldrm import Classifier
from datasets import  final_extinctionrisk_dataframe, final_extinctionrisk_noth_dataframe

from algo import prune_rules

In [2]:
random.seed(42)

# With Human Threats

In [3]:
model_template, data = final_extinctionrisk_dataframe()
X = data[model_template.attrs].drop('Extinction_risk',axis=1)
X = X.convert_dtypes()
X = X.to_numpy()
y = data['Extinction_risk'].to_numpy()

# Loading the Train/Test Splits from R

In [4]:
import csv

train_indices = []
test_indices = []
for i in range(1,11):
    with open("./datasets/r_folds/fold_"+str(i)+"_train_idx.csv", newline='') as csvfile:
        train_df = pd.read_csv(csvfile)
        train_list = train_df.to_numpy().flatten().tolist()
        train_indices.append(train_list)
    with open("./datasets/r_folds/fold_"+str(i)+"_test_idx.csv", newline='') as csvfile:
        test_df = pd.read_csv(csvfile)
        test_list = test_df.to_numpy().flatten().tolist()
        test_indices.append(test_list)

# K-Fold Cross Validation With Human Threats.
- Data Driven Model

In [5]:
acc_metrics = []
f1_metrics = [] 
precision_metrics = []
recall_metrics = []


for i, (train_index, test_index) in enumerate(zip(train_indices, test_indices)):
    print("Fold "+str(i)+":")
    
    train_data = np.concatenate((np.array(X[train_index]), np.array(y[train_index]).reshape(-1, 1)), axis=1).tolist()
    test_data = np.concatenate((np.array(X[test_index]), np.array(y[test_index]).reshape(-1, 1)), axis=1).tolist()
 
    # Prepare the test data (features and true labels)
    X_test = [d[:-1] for d in test_data]
    Y_test = [d[-1] for d in test_data]

    advanced_pruning_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)

    # Now, train using confidence_fit with a high 15% improvement threshold
    advanced_pruning_model.confidence_fit(train_data, improvement_threshold=0.20)
    advanced_pruning_model.print_asp(simple=True)


    adv_pruning_predictions = advanced_pruning_model.predict(X_test)
    adv_pruning_labels = [p[0] for p in adv_pruning_predictions]
    for i in range(len(adv_pruning_labels)):
        if adv_pruning_labels[i] is None:
            adv_pruning_labels[i] = ('None')

    acc = accuracy_score(Y_test, adv_pruning_labels)

    f1 = f1_score(Y_test, adv_pruning_labels, average=None, labels=["Higher_risk", "Lower_risk"]) #["Higher_risk", "Lower_risk"]["Lower_risk", "Higher_risk"]
    precision = precision_score(Y_test,adv_pruning_labels, average=None, labels=["Higher_risk", "Lower_risk"])
    recall = recall_score(Y_test, adv_pruning_labels, average=None, labels=["Higher_risk", "Lower_risk"])
    
    acc_metrics.append(acc)    
    f1_metrics.append(f1)
    precision_metrics.append(precision)
    recall_metrics.append(recall)

    print("----------------------------------------")
    print()

Fold 0:
extinction_risk(X,'lower_risk') :- hunting(X,N3), N3<=0. [confidence: 0.94723]
extinction_risk(X,'higher_risk') :- range_size(X,N25), N25<=11549.62. [confidence: 0.86842]
extinction_risk(X,'higher_risk') :- agriculture(X,N2), N2>0, range_size(X,N25), N25<=5118144.61. [confidence: 0.67252]
extinction_risk(X,'lower_risk') :- maximum_latitude(X,N13), N13>65.94. [confidence: 0.81496]
extinction_risk(X,'lower_risk') :- adult_survival_annual(X,N23), N23<=0.86675, range_size(X,N25), N25>6168290.72. [confidence: 0.80814]
extinction_risk(X,'higher_risk') :- invasive_species(X,N4), N4>0. [confidence: 0.67708]
extinction_risk(X,'lower_risk') :- tarsus_length(X,N8), N8<=91.6. [confidence: 0.65951]
extinction_risk(X,'higher_risk') :- clutch_size(X,N27), N27<=3.73214. [confidence: 0.78333]
extinction_risk(X,'lower_risk') :- minimum_latitude(X,N12), N12<=26.25. [confidence: 0.67647]
extinction_risk(X,'higher_risk') :- order(X,'accipitriformes'). [confidence: 0.59091]
-------------------------

In [6]:
print("Data Driven Model")
print(f"KFold Mean Acc: {round(np.mean(acc_metrics), 5)} | KFold SD: {np.std(acc_metrics)}")
f1_metrics = np.array(f1_metrics)
precision_metrics = np.array(precision_metrics)
recall_metrics = np.array(recall_metrics)

print(f"KFold Mean Precision: Higher Risk {round(np.mean(precision_metrics[:,0]), 5)}, Lower Risk {round(np.mean(precision_metrics[:,1]), 5)}  | KFold SD: {round(np.std(precision_metrics[:, 0]),5), round(np.std(precision_metrics[:, 1]), 5)}")
print(f"KFold Mean Recall: Higher Risk {round(np.mean(recall_metrics[:,0]), 5)}, Lower Risk {round(np.mean(recall_metrics[:,1]), 5)} | KFold SD: {round(np.std(recall_metrics[:, 0]), 5), round(np.std(recall_metrics[:, 1]), 5)}")
print(f"KFold Mean f1: Higher Risk {round(np.mean(f1_metrics[:,0]), 5)}, Lower Risk {round(np.mean(f1_metrics[:,1]), 5)} | KFold SD: {round(np.std(f1_metrics[:,0]), 5), round(np.std(f1_metrics[:,1]), 5)}")

Data Driven Model
KFold Mean Acc: 0.89771 | KFold SD: 0.007748123584739406
KFold Mean Precision: Higher Risk 0.76374, Lower Risk 0.92493  | KFold SD: (np.float64(0.03166), np.float64(0.00516))
KFold Mean Recall: Higher Risk 0.67129, Lower Risk 0.95089 | KFold SD: (np.float64(0.02486), np.float64(0.0083))
KFold Mean f1: Higher Risk 0.71402, Lower Risk 0.93771 | KFold SD: (np.float64(0.02019), np.float64(0.00484))


# K-Fold Cross Validation With Human Threats.
- Expert Rule Model

In [7]:
acc_metrics = []
f1_metrics = [] 
precision_metrics = []
recall_metrics = []


for i, (train_index, test_index) in enumerate(zip(train_indices, test_indices)):
    print("Fold "+str(i)+":")
    
    train_data = np.concatenate((np.array(X[train_index]), np.array(y[train_index]).reshape(-1, 1)), axis=1).tolist()
    test_data = np.concatenate((np.array(X[test_index]), np.array(y[test_index]).reshape(-1, 1)), axis=1).tolist()
 
    # Prepare the test data (features and true labels)
    X_test = [d[:-1] for d in test_data]
    Y_test = [d[-1] for d in test_data]

    # Instantiate a new classifier for expert-guided model
    expert_model = Classifier(attrs=model_template.attrs.copy(), numeric=model_template.numeric, label=model_template.label)

    # Define our expert rules as strings
    rule1 = "with confidence 0.99 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Agriculture' '==' '1' and 'Invasive_species' '==' '1' and 'Hunting' '==' '1'"
    rule2 = "with confidence 0.95 class = 'Higher_risk' if 'Agriculture' '==' '1' and 'Invasive_species' '==' '1' and 'Hunting' '==' '1'"
    rule3 = "with confidence 0.95 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Invasive_species' '==' '1' and 'Hunting' '==' '1'"
    rule4 = "with confidence 0.95 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Agriculture' '==' '1' and 'Hunting' '==' '1'"
    rule5 = "with confidence 0.95 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Agriculture' '==' '1' and 'Invasive_species' '==' '1'"

    # Add the manual rules to the model
    expert_model.add_manual_rule(rule1, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
    expert_model.add_manual_rule(rule2, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
    expert_model.add_manual_rule(rule3, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
    expert_model.add_manual_rule(rule4, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
    expert_model.add_manual_rule(rule5, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)

    #expert_model.fit(train_data, ratio=0.75)

    expert_model.confidence_fit(train_data, improvement_threshold=0.20)
    expert_model.print_asp(simple=True)


    expert_predictions_tuples = expert_model.predict(X_test)
    expert_predicted_labels = [p[0] for p in expert_predictions_tuples]
    for i in range(len(expert_predicted_labels)):
        if expert_predicted_labels[i] is None:
            expert_predicted_labels[i] = ('None')

    acc = accuracy_score(Y_test, expert_predicted_labels)

    f1 = f1_score(Y_test, expert_predicted_labels, average=None, labels=["Higher_risk", "Lower_risk"]) #["Higher_risk", "Lower_risk"]["Lower_risk", "Higher_risk"]
    precision = precision_score(Y_test,expert_predicted_labels, average=None, labels=["Higher_risk", "Lower_risk"])
    recall = recall_score(Y_test, expert_predicted_labels, average=None, labels=["Higher_risk", "Lower_risk"])
    
    acc_metrics.append(acc)    
    f1_metrics.append(f1)
    precision_metrics.append(precision)
    recall_metrics.append(recall)

    print("----------------------------------------")
    print()

Fold 0:
extinction_risk(X,'higher_risk') :- agriculture(X,N2), N2>1, hunting(X,N3), N3>1, invasive_species(X,N4), N4>1, range_size(X,N25), N25<=75321. [confidence: 0.99]
extinction_risk(X,'higher_risk') :- agriculture(X,N2), N2>1, hunting(X,N3), N3>1, invasive_species(X,N4), N4>1. [confidence: 0.95]
extinction_risk(X,'higher_risk') :- hunting(X,N3), N3>1, invasive_species(X,N4), N4>1, range_size(X,N25), N25<=75321. [confidence: 0.95]
extinction_risk(X,'higher_risk') :- agriculture(X,N2), N2>1, hunting(X,N3), N3>1, range_size(X,N25), N25<=75321. [confidence: 0.95]
extinction_risk(X,'higher_risk') :- agriculture(X,N2), N2>1, invasive_species(X,N4), N4>1, range_size(X,N25), N25<=75321. [confidence: 0.95]
extinction_risk(X,'lower_risk') :- hunting(X,N3), N3<=0. [confidence: 0.9563]
extinction_risk(X,'higher_risk') :- realm(X,'i'). [confidence: 0.80636]
extinction_risk(X,'lower_risk') :- maximum_latitude(X,N13), N13>65.53. [confidence: 0.80804]
extinction_risk(X,'higher_risk') :- climate_ch

In [8]:
print("Expert Rule Model")
print(f"KFold Mean Acc: {round(np.mean(acc_metrics), 5)} | KFold SD: {np.std(acc_metrics)}")
f1_metrics = np.array(f1_metrics)
precision_metrics = np.array(precision_metrics)
recall_metrics = np.array(recall_metrics)

print(f"KFold Mean Precision: Higher Risk {round(np.mean(precision_metrics[:,0]), 5)}, Lower Risk {round(np.mean(precision_metrics[:,1]), 5)}  | KFold SD: {round(np.std(precision_metrics[:, 0]),5), round(np.std(precision_metrics[:, 1]), 5)}")
print(f"KFold Mean Recall: Higher Risk {round(np.mean(recall_metrics[:,0]), 5)}, Lower Risk {round(np.mean(recall_metrics[:,1]), 5)} | KFold SD: {round(np.std(recall_metrics[:, 0]), 5), round(np.std(recall_metrics[:, 1]), 5)}")
print(f"KFold Mean f1: Higher Risk {round(np.mean(f1_metrics[:,0]), 5)}, Lower Risk {round(np.mean(f1_metrics[:,1]), 5)} | KFold SD: {round(np.std(f1_metrics[:,0]), 5), round(np.std(f1_metrics[:,1]), 5)}")

Expert Rule Model
KFold Mean Acc: 0.90324 | KFold SD: 0.008232344762685827
KFold Mean Precision: Higher Risk 0.7547, Lower Risk 0.93708  | KFold SD: (np.float64(0.02922), np.float64(0.00605))
KFold Mean Recall: Higher Risk 0.73001, Lower Risk 0.94394 | KFold SD: (np.float64(0.02739), np.float64(0.009))
KFold Mean f1: Higher Risk 0.74164, Lower Risk 0.94046 | KFold SD: (np.float64(0.02047), np.float64(0.0052))


# Training on Complete Dataset 
- withought Expert Rules 
- and with Human Threats

In [9]:
baseline_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
train_data = np.concatenate((np.array(X), np.array(y).reshape(-1, 1)), axis=1).tolist()

advanced_pruning_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
advanced_pruning_model.confidence_fit(train_data, improvement_threshold=0.20)
advanced_pruning_model.print_asp(simple=True)

extinction_risk(X,'lower_risk') :- agriculture(X,N2), N2<=0. [confidence: 0.94195]
extinction_risk(X,'higher_risk') :- range_size(X,N25), N25<=13551.2. [confidence: 0.87655]
extinction_risk(X,'higher_risk') :- not realm(X,'l'), range_size(X,N25), N25<=5136924.07. [confidence: 0.70704]
extinction_risk(X,'lower_risk') :- tail_length(X,N11), N11<=304.0, maximum_latitude(X,N13), N13>1.03, range_size(X,N25), N25<=19212039.47, body_mass(X,N26), N26<=79.2. [confidence: 0.77899]
extinction_risk(X,'lower_risk') :- range_size(X,N25), N25>19212039.47. [confidence: 0.89024]
extinction_risk(X,'higher_risk') :- tail_length(X,N11), N11>304.0. [confidence: 0.82258]
extinction_risk(X,'lower_risk') :- not realm(X,'p'), maximum_elevation(X,N22), N22>1600.0, adult_survival_annual(X,N23), N23<=0.87484, range_size(X,N25), N25>354187.36. [confidence: 0.75]
extinction_risk(X,'higher_risk') :- not realm(X,'p'), beak_depth(X,N7), N7>3.3, hand_wing_index(X,N10), N10>18.9. [confidence: 0.68155]
extinction_risk(X,

# Training on Complete Dataset 
- With Expert Rules 
- and with Human Threats

In [10]:
train_data = np.concatenate((np.array(X), np.array(y).reshape(-1, 1)), axis=1).tolist()

expert_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)

# Define our expert rules as strings
rule1 = "with confidence 0.99 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Agriculture' '==' '1' and 'Invasive_species' '==' '1' and 'Hunting' '==' '1'"
rule2 = "with confidence 0.95 class = 'Higher_risk' if 'Agriculture' '==' '1' and 'Invasive_species' '==' '1' and 'Hunting' '==' '1'"
rule3 = "with confidence 0.95 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Invasive_species' '==' '1' and 'Hunting' '==' '1'"
rule4 = "with confidence 0.95 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Agriculture' '==' '1' and 'Hunting' '==' '1'"
rule5 = "with confidence 0.95 class = 'Higher_risk' if 'Range_size' '<=' '75321' and 'Agriculture' '==' '1' and 'Invasive_species' '==' '1'"

# Add the manual rules to the model
expert_model.add_manual_rule(rule1, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
expert_model.add_manual_rule(rule2, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
expert_model.add_manual_rule(rule3, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
expert_model.add_manual_rule(rule4, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)
expert_model.add_manual_rule(rule5, model_template.attrs, model_template.numeric, ['Lower_risk', 'Higher_risk'], instructions=False)

#expert_model.fit(train_data, ratio=0.75)

expert_model.confidence_fit(train_data, improvement_threshold=0.20)
expert_model.print_asp(simple=True)

extinction_risk(X,'higher_risk') :- agriculture(X,N2), N2>1, hunting(X,N3), N3>1, invasive_species(X,N4), N4>1, range_size(X,N25), N25<=75321. [confidence: 0.99]
extinction_risk(X,'higher_risk') :- agriculture(X,N2), N2>1, hunting(X,N3), N3>1, invasive_species(X,N4), N4>1. [confidence: 0.95]
extinction_risk(X,'higher_risk') :- hunting(X,N3), N3>1, invasive_species(X,N4), N4>1, range_size(X,N25), N25<=75321. [confidence: 0.95]
extinction_risk(X,'higher_risk') :- agriculture(X,N2), N2>1, hunting(X,N3), N3>1, range_size(X,N25), N25<=75321. [confidence: 0.95]
extinction_risk(X,'higher_risk') :- agriculture(X,N2), N2>1, invasive_species(X,N4), N4>1, range_size(X,N25), N25<=75321. [confidence: 0.95]
extinction_risk(X,'lower_risk') :- hunting(X,N3), N3<=0. [confidence: 0.95533]
extinction_risk(X,'higher_risk') :- realm(X,'i'). [confidence: 0.80256]
extinction_risk(X,'lower_risk') :- maximum_latitude(X,N13), N13>65.53. [confidence: 0.80894]
extinction_risk(X,'higher_risk') :- climate_change(X,